## Udacity Steering Angle Prediction Dataset

The [Udacity Self driving Car](https://github.com/udacity/self-driving-car) is an open source project aimed at developing DL systems for various aspects of a self driving car. The datasets are avaiable to download from the [Github repository](https://github.com/udacity/self-driving-car/tree/master/datasets).




In [1]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from PIL import Image

#### Parameters and Variables

In [2]:
import os
from pathlib import Path



# Model params
img_x, img_y = 32, 32
resize_shape = (img_x, img_y)
batch_size = 1000
num_epochs = 15
num_frames = 30
learning_rate = 1e-4
gamma=0.1
milestones = [0.5 * num_epochs, 0.75 * num_epochs]


# the min angle is -0.4 and the max angle is 0.4, we try to divide it into 3 classes
num_classes = 3 
num_channels = 3 # RGB

# 3D CNN parameters
fc_hidden1, fc_hidden2 = 32, 32
dropout = 0.5       # dropout probability


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Path
PWD = Path(os.path.abspath('')).parent.resolve()
MODELS_PATH = Path(os.path.abspath('')).parent.resolve() / "models/"
DATASET_PATH = "/vol/bitbucket/svadakku/data/udacity/Ch2_001"
# TEST_DATASET_PATH = "/vol/bitbucket/svadakku/data/udacity/Challenge2/Test"

In [3]:

# Dataset to convert Udacity images into sequence of frames
class UdacitySteeringAngleDataset(Dataset):
    """Custom Dataset for loading images and steering angles for Udacity self-driving car dataset."""
    
    def __init__(self, image_dir, labels_file, transform=None, train=True, test_size=0.2, random_state=42, resize_shape=(img_x, img_y), num_classes=10, num_frames=30):
        """
        Args:
            image_dir (str): Directory containing all the images.
            labels_file (str): Path to CSV file with image filenames and steering angles.
            transform (callable, optional): Optional transform to be applied on a sample.
            train (bool): training or testing split
            test_size (float): Proportion of the dataset to be used as test set.
            random_state (int): Random seed for reproducibility.
        """
        self.image_dir = image_dir
        self.labels_file = labels_file
        
        self.train = train
        self.img_x = img_x
        self.img_y = img_y
        self.num_classes = num_classes
        self.num_frames = num_frames
        self.thresholds = np.linspace(-1.0, 1.0, num_classes + 1)
        
        
        # Define the transform to be applied to the images
        # If transform is not provided, use default transformations
        if transform is not None:
            self.transform = transform
        else:
            train_transform = transforms.Compose([
                transforms.Resize((img_x, img_y)),  # Typical size for Udacity dataset
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Add some color variation
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Small random shifts
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            
            test_transform = transforms.Compose([
                transforms.Resize((img_x, img_y)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
            if train:
                # Define transforms for steering angle prediction task
                self.transform = train_transform
            else:
                self.transform = test_transform
                
        
        # Read the CSV file with image filenames and steering angles
        self.data_frame = pd.read_csv(labels_file)
        
        # Get image paths and steering angles
        self.image_files = [os.path.join(image_dir, f"{filename}.jpg") for filename in self.data_frame['frame_id'].values]
        self.steering_angles = self.data_frame['steering_angle'].values.astype(np.float32)
        
        # Split the dataset into train and test sets
        train_files, test_files, train_angles, test_angles = train_test_split(
            self.image_files, self.steering_angles, test_size=test_size, random_state=random_state
        )
        
        # Select the appropriate data split
        if train:
            self.image_files = train_files
            self.steering_angles = train_angles
        else:  # 'test' split
            self.image_files = test_files
            self.steering_angles = test_angles
    
    
    def __len__(self):
        """Return the total number of images."""
        return len(self.image_files)
    
    
    def __getitem__(self, idx):
        
        start_idx = idx
        if (start_idx + self.num_frames  >= len(self.image_files)):
            # If the index is too close to the end, we need to backtrack
            start_idx = len(self.image_files) - self.num_frames - 1
        
        
        # Load images
        images = []
        angle_labels = []
        for i in range(start_idx, start_idx+self.num_frames):
            img_path = self.image_files[idx]
            steering_angle = self.steering_angles[idx]
            img = Image.open(img_path)
            if self.transform:
                img = self.transform(img)
            images.append(img)
            # Get steering angle and convert to class
            angle_labels.append(self._get_steering_angle_class_(steering_angle))
        
        # Stack images into tensor
        image_sequence = torch.stack(images, dim=1) # Shape: (channels, depth, W, H)
        
        
        steering_class = max(set(angle_labels), key=angle_labels.count)
        
        return image_sequence, steering_class


    def _get_steering_angle_class_(self, angle):
        """
        Convert continuous steering angles to discrete classes.
        
        Classes example (10 classes):
        - Extreme Left:  [-1.0 to -0.8]   -> 0
        - Hard Left:     [-0.8 to -0.6]   -> 1
        - Medium Left:   [-0.6 to -0.4]   -> 2
        - Soft Left:     [-0.4 to -0.2]   -> 3
        - Slight Left:   [-0.2 to 0.0]    -> 4
        - Slight Right:  [0.0 to 0.2]     -> 5
        - Soft Right:    [0.2 to 0.4]     -> 6
        - Medium Right:  [0.4 to 0.6]     -> 7
        - Hard Right:    [0.6 to 0.8]     -> 8
        - Extreme Right: [0.8 to 1.0]     -> 9
        
        Classes example (3 classes):
        - Left: [-1.0 to -0.333] -> 0
        - Straight:[-0.333 to 0.3333] -> 1
        - Right: [-0.333 to 1]   -> 2
        """
        angle = np.clip(angle, -1.0, 1.0)  # Ensure angle is within bounds
        for i in range(len(self.thresholds) - 1):
            if self.thresholds[i] <= angle <= self.thresholds[i + 1]:
                return i
        return self.num_classes - 1  # For edge case of angle = 1.0
    
    

# Load training and validation datasets
train_dataset = UdacitySteeringAngleDataset(image_dir=f"{DATASET_PATH}/center", labels_file=f"{DATASET_PATH}/final_example.csv", train=True, resize_shape=resize_shape, num_classes=num_classes, num_frames=num_frames)
val_dataset = UdacitySteeringAngleDataset(image_dir=f"{DATASET_PATH}/center", labels_file=f"{DATASET_PATH}/final_example.csv", train=False, resize_shape=resize_shape, num_classes=num_classes, num_frames=num_frames)

train_dataset_at_eval = UdacitySteeringAngleDataset(image_dir=f"{DATASET_PATH}/center", labels_file=f"{DATASET_PATH}/final_example.csv", train=True, resize_shape=resize_shape, num_classes=num_classes)
# Set the transform of the training eval to be the same as the validation dataset
train_dataset_at_eval.transform = val_dataset.transform

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
train_loader_at_eval = DataLoader(train_dataset_at_eval, batch_size=batch_size, shuffle=False)

#### 3DCNN Model

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def conv3D_output_size(img_size, padding, kernel_size, stride):
    # compute output shape of conv3D
    outshape = (np.floor((img_size[0] + 2 * padding[0] - (kernel_size[0] - 1) - 1) / stride[0] + 1).astype(int),
                np.floor((img_size[1] + 2 * padding[1] - (kernel_size[1] - 1) - 1) / stride[1] + 1).astype(int),
                np.floor((img_size[2] + 2 * padding[2] - (kernel_size[2] - 1) - 1) / stride[2] + 1).astype(int))
    return outshape

class CNN3D(nn.Module):
    def __init__(self, t_dim=120, img_x=90, img_y=120, drop_p=0.2, fc_hidden1=256, fc_hidden2=128, num_classes=50, in_channels=1):
        super(CNN3D, self).__init__()
        
        self.layers = []
        self.batch_norms = []        
        # set video dimension
        self.t_dim = t_dim
        self.img_x = img_x
        self.img_y = img_y
        # fully connected layer hidden nodes
        self.fc_hidden1, self.fc_hidden2 = fc_hidden1, fc_hidden2
        self.drop_p = drop_p
        self.num_classes = num_classes
        self.ch1, self.ch2 = 32, 48
        self.k1, self.k2 = (5, 5, 5), (3, 3, 3)  # 3d kernel size
        self.s1, self.s2 = (2, 2, 2), (2, 2, 2)  # 3d strides
        self.pd1, self.pd2 = (0, 0, 0), (0, 0, 0)  # 3d padding

        # compute conv1 & conv2 output shape
        self.conv1_outshape = conv3D_output_size((self.t_dim, self.img_x, self.img_y), self.pd1, self.k1, self.s1)
        self.conv2_outshape = conv3D_output_size(self.conv1_outshape, self.pd2, self.k2, self.s2)

        self.conv1 = nn.Conv3d(in_channels=in_channels, out_channels=self.ch1, kernel_size=self.k1, stride=self.s1,
                               padding=self.pd1)
        self.layers.append("Conv3d_2_0")
        
        self.bn1 = nn.BatchNorm3d(self.ch1)
        self.layers.append("BatchNorm3d_32")
        self.batch_norms.append(self.bn1)
        
        self.layers.append("ReLU")
        
        self.conv2 = nn.Conv3d(in_channels=self.ch1, out_channels=self.ch2, kernel_size=self.k2, stride=self.s2,
                               padding=self.pd2)
        self.layers.append("Conv3d_2_0")
        
        self.bn2 = nn.BatchNorm3d(self.ch2)
        self.layers.append("BatchNorm3d_48")
        self.batch_norms.append(self.bn2)
        
        self.relu = nn.ReLU(inplace=True)
        self.layers.append("ReLU")
        
        self.drop = nn.Dropout3d(self.drop_p)
        self.pool = nn.MaxPool3d(2)
        
        # Add adaptive pooling layer to handle different input sizes
        self.adaptive_pool = nn.AdaptiveAvgPool3d((2, 2, 2))
        
        self.layers.append("Flatten")
        # Fixed size output from adaptive pooling: ch2 * 2 * 2 * 2
        fc_input_size = self.ch2 * 2 * 2 * 2
        
        self.fc1 = nn.Linear(fc_input_size, self.fc_hidden1)
        self.layers.append("Linear")
        self.layers.append("ReLU")
        
        self.fc2 = nn.Linear(self.fc_hidden1, self.fc_hidden2)
        self.layers.append("Linear")
        self.layers.append("ReLU")
        
        self.fc3 = nn.Linear(self.fc_hidden2, self.num_classes)  # fully connected layer, output = multi-classes
        self.layers.append("Linear")
        
    def forward(self, x_3d):
        # Conv 1
        x = self.conv1(x_3d)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.drop(x)
        
        # Conv 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.drop(x)
        
        # Use adaptive pooling to get fixed size output regardless of input dimensions
        x = self.adaptive_pool(x)
        
        # FC 1 and 2
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=self.drop_p, training=self.training)
        x = self.fc3(x)
        
        return x 
    

class UdacitySteeringAngle_3DCNN(nn.Module):
    def __init__(self, num_classes=3, in_channels=3):
        super(UdacitySteeringAngle_3DCNN, self).__init__()
        self.num_classes = num_classes
        self.num_frames = num_frames
        self.img_x = img_x
        self.img_y = img_y
        
        # Define the 3D CNN model
        self.cnn3d = CNN3D(t_dim=num_frames, img_x=img_x, img_y=img_y, fc_hidden1=fc_hidden1, fc_hidden2=fc_hidden2, num_classes=num_classes, in_channels=in_channels, drop_p=dropout)
        
    def forward(self, x):
        return self.cnn3d(x)

#### Training

In [5]:
from sklearn import metrics


def train(model, train_loader, criterion, optimizer, device):
    total_loss = []

    model.train()
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        inputs = inputs.to(device)
        outputs = model(inputs.to(device))

        targets = targets.to(device)
        # targets = torch.squeeze(targets, 1).long().to(device)
        loss = criterion(outputs, targets)

        total_loss.append(loss.item())

        loss.backward()
        optimizer.step()
    
    epoch_loss = sum(total_loss)/len(total_loss)
    return epoch_loss


def test(model, data_loader, criterion, device):

    model.eval()

    total_loss = []
    all_predictions = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(data_loader):
            inputs = inputs.to(device)
            outputs = model(inputs.to(device))
        
            targets = targets.to(device)
            # targets = torch.squeeze(targets, 1).long().to(device)
            loss = criterion(outputs, targets)
            
            # m = nn.Softmax(dim=1)
            # probs = m(outputs).to(device)
            # targets = targets.float().resize_(len(targets), 1)
            preds = torch.argmax(outputs, dim=1)
            
            all_predictions.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            # all_probs.extend(probs.cpu().numpy())
            total_loss.append(loss.item())
        
         # Convert to numpy arrays
        all_predictions = np.array(all_predictions)
        all_targets = np.array(all_targets)
        
        # Calculate metrics
        acc = metrics.accuracy_score(all_targets, all_predictions)
        # auc = metrics.roc_auc_score(
        #     np.eye(model.fc.out_features)[all_targets],  # One-hot encode targets
        #     all_probs,
        #     multi_class='ovr'
        # )

        test_loss = sum(total_loss) / len(total_loss)

        return [test_loss, acc] # , auc]


In [7]:
from copy import deepcopy
from collections import OrderedDict
from tqdm import trange

device = torch.device(DEVICE)

model = UdacitySteeringAngle_3DCNN(num_classes=num_classes, in_channels=num_channels)

# Parallelize model to multiple GPUs
# if device == 'cuda' and torch.cuda.device_count() > 1:
#     print("Using", torch.cuda.device_count(), "GPUs!")
#     model = nn.DataParallel(model)

model = model.to(device)


criterion = nn.CrossEntropyLoss()


optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)
    
logs = ['loss', 'acc'] #, 'auc']
train_logs = ['train_'+log for log in logs]
# val_logs = ['val_'+log for log in logs]
test_logs = ['test_'+log for log in logs]
# log_dict = OrderedDict.fromkeys(train_logs+val_logs+test_logs, 0)
log_dict = OrderedDict.fromkeys(train_logs+test_logs, 0)
    

best_acc = 0
best_epoch = 0
best_model = deepcopy(model)


for epoch in trange(num_epochs):
    
    # print(f"Epoch: {epoch} / {num_epochs}")
    train_loss = train(model, train_loader, criterion, optimizer, device)
    
    train_metrics = test(model, train_loader_at_eval, criterion, device)
    # val_metrics = test(model, val_evaluator, val_loader, criterion, device, run)
    test_metrics = test(model, val_loader, criterion, device)
    
    scheduler.step()
    
    for i, key in enumerate(train_logs):
        log_dict[key] = train_metrics[i]
    # for i, key in enumerate(val_logs):
    #     log_dict[key] = val_metrics[i]
    for i, key in enumerate(test_logs):
        log_dict[key] = test_metrics[i]

        
    # cur_auc = val_metrics[1]
    cur_acc = test_metrics[1]
    if cur_acc > best_acc:
        best_epoch = epoch
        best_acc = cur_acc
        best_model = deepcopy(model)
        print('curr acc', cur_acc)
        print('cur_best_acc:', best_acc)
        print('cur_best_epoch', best_epoch)

state = {
    'net': model.state_dict(),
}

path = os.path.join(MODELS_PATH, 'udacity_3dcnn_32_32_30.pth')
torch.save(model.state_dict(), path)
print(f"Model saved to {path}")

train_metrics = test(best_model, train_loader_at_eval, criterion, device)
# val_metrics = test(best_model, val_evaluator, val_loader, criterion, device, run, MODELS_PATH)
test_metrics = test(best_model, val_loader, criterion, device)

train_log = 'train  acc: %.5f  loss: %.5f\n' % (train_metrics[1], train_metrics[0])
# val_log = 'val  auc: %.5f  acc: %.5f\n' % (val_metrics[1], val_metrics[2])
test_log = 'test  acc: %.5f  loss: %.5f\n' % (test_metrics[1], test_metrics[0])

# log = '%s\n' % (data_flag) + train_log + val_log + test_log + '\n'
log = '%s\n' % 'udacity_steering_angle' + train_log + test_log + '\n'
print(log)
print('cur_best_epoch', best_epoch)


with open(os.path.join(MODELS_PATH, '%s_log.txt' % ('udacity_steering_angle')), 'a') as f:
        f.write(log)  

  0%|          | 0/15 [00:00<?, ?it/s]

  7%|▋         | 1/15 [18:25<4:17:57, 1105.52s/it]

curr acc 0.07658058771148708
cur_best_acc: 0.07658058771148708
cur_best_epoch 0


 20%|██        | 3/15 [55:12<3:40:49, 1104.13s/it]

curr acc 0.09795191451469279
cur_best_acc: 0.09795191451469279
cur_best_epoch 2


...

curr acc 0.8414959928762245
cur_best_acc: 0.8414959928762245
cur_best_epoch 8


 67%|██████▋   | 10/15 [3:02:45<1:30:29, 1085.82s/it]

curr acc 0.8423864648263579
cur_best_acc: 0.8423864648263579
cur_best_epoch 9


100%|██████████| 15/15 [4:33:00<00:00, 1092.03s/it]  


Model saved to /data/sv/temporal-nn-verification/udacity_steering_angle_prediction/models/udacity_3dcnn_32_32_30.pth
udacity_steering_angle
train  acc: 0.84792  loss: 0.87893
test  acc: 0.84239  loss: 0.88659


cur_best_epoch 9


##### 